# Fase 1 — Aquisição Sentinel-2 (GEE)

Aquisição de imagens **Sentinel-2 Nível-2A** (bandas B2, B3, B4 e B8 a 10 m) para a área de estudo — Região Geográfica Imediata de Guaxupé - MG (código IBGE 310044).

Este estágio importa a malha vetorial do IBGE, define o polígono da área de estudo, monta a coleção filtrada por data e cobertura de nuvem, gera a composição livre de nuvens (Cloud Score+ com fallback QA60) e dispara a exportação dos GeoTIFF para o Google Drive. As saídas são os mosaicos recortados e as tarefas de exportação registradas.

## Detecção da raiz do repositório

Localiza a raiz do repositório a partir do diretório corrente e a insere no caminho de importação, garantindo o acesso ao pacote `src/`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


# Sobe os diretórios até encontrar src/config.yaml, marcador da raiz do projeto.
def _find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "config.yaml").is_file():
            return candidate
    raise RuntimeError("Raiz do repositório não localizada (src/config.yaml ausente).")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Raiz do projeto: {PROJECT_ROOT}")

## Detecção da plataforma

Identifica o ambiente de execução (Kaggle, Colab ou local) para adaptar a instalação de dependências e a leitura de segredos.

In [ ]:
import importlib.util
import os


# Heurística por variáveis de ambiente e presença de diretórios característicos.
def detect_platform() -> str:
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").is_dir():
        return "kaggle"
    if "COLAB_GPU" in os.environ or importlib.util.find_spec("google.colab") is not None:
        return "colab"
    return "local"


PLATFORM = detect_platform()
print(f"Plataforma detectada: {PLATFORM}")

## Instalação condicional das dependências

Em Kaggle/Colab instala o pacote com os extras geoespaciais e de aprendizado de máquina. No ambiente local a instalação é ignorada, pois é gerenciada por `uv` e pelo CI.

In [ ]:
import subprocess

# Instala o projeto editavelmente com os extras necessários apenas em nuvem.
if PLATFORM in {"kaggle", "colab"}:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[geo,ml]"],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print("Dependências instaladas.")
else:
    print("Ambiente local: instalação ignorada (gerenciada por uv/CI).")

## Carregamento da configuração única

Lê a configuração de `src/config.yaml` por meio de `src/config.py`, fonte única de verdade de caminhos, bandas, parâmetros e sementes.

In [ ]:
from src.config import CONFIG

# Exibe os parâmetros de aquisição definidos na configuração.
print(f"Área de estudo: {CONFIG.get('aoi.region_name')} ({CONFIG.get('aoi.region_code')})")
print(f"Coleção: {CONFIG.get('gee.image_collection')}")
print(f"Janela: {CONFIG.get('gee.start_date')} a {CONFIG.get('gee.end_date')}")
print(f"Bandas: {CONFIG.bands} | Escala: {CONFIG.get('gee.scale_m')} m")

## Fixação das sementes

Fixa as sementes de `python`, `numpy`, `torch` e `cuda` para garantir a reprodutibilidade das etapas amostrais (ex.: clusterização).

In [ ]:
from src.config import seed_everything

# Aplica a semente global definida na configuração.
resolved_seed = seed_everything()
print(f"Sementes fixadas em {resolved_seed}.")

## Carregamento de segredos

Em Kaggle/Colab injeta os segredos do cofre da plataforma nas variáveis de ambiente esperadas pelo pacote. Nenhum valor é impresso. No ambiente local, os segredos devem vir de variáveis de ambiente ou do arquivo `.env`.

In [ ]:
# Nomes das variáveis de ambiente consumidas pela aquisição.
SECRET_NAMES = (
    "GEE_SERVICE_ACCOUNT_EMAIL",
    "GEE_PROJECT",
    "GEE_SERVICE_ACCOUNT_KEY_JSON",
)

if PLATFORM == "kaggle":
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()
    for name in SECRET_NAMES:
        try:
            os.environ[name] = client.get_secret(name)
        except Exception:
            print(f"Segredo ausente no Kaggle: {name}")
elif PLATFORM == "colab":
    from google.colab import userdata

    for name in SECRET_NAMES:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            print(f"Segredo ausente no Colab: {name}")
else:
    print("Ambiente local: segredos esperados via variáveis de ambiente/.env.")

## Autenticação no Google Earth Engine

Inicializa o Earth Engine com a conta de serviço lida exclusivamente do ambiente e interrompe a execução caso as credenciais estejam ausentes, pois toda a fase depende do serviço.

In [ ]:
from src.data.gee_client import init_ee

# Inicializa o cliente do Earth Engine; sem credenciais a fase não prossegue.
ee = init_ee()
print("Earth Engine autenticado com sucesso.")

## Carregamento da área de estudo

Carrega a malha vetorial do IBGE e recorta o registro da Região Geográfica Imediata de Guaxupé, convertendo a geometria para o formato do Earth Engine.

In [ ]:
from src.data.aoi import geometry_bounds, geometry_to_ee, get_region_geometry

# Obtém a geometria unificada da região e a converte para ee.Geometry.
aoi_geometry = get_region_geometry()
aoi_ee = geometry_to_ee(ee, aoi_geometry)
print(f"Limites (minx, miny, maxx, maxy): {geometry_bounds(aoi_geometry)}")

## Montagem da coleção Sentinel-2

Monta a coleção Sentinel-2 Nível-2A filtrada pela área de estudo, pela janela temporal e pelo percentual máximo de nuvem, usando os parâmetros da configuração.

In [ ]:
from src.data.gee_client import get_s2_sr_collection

# Filtra a coleção por área, data e cobertura de nuvem.
collection = get_s2_sr_collection(ee, aoi_ee)
collection_size = collection.size().getInfo()
print(f"Cenas Sentinel-2 disponíveis: {collection_size}")

## Composição livre de nuvens

Aplica a máscara Cloud Score+ (com fallback QA60) e reduz a série temporal em uma composição pela mediana, recortada pela área de estudo.

In [ ]:
from src.data.gee_client import build_cloud_free_composite

# Gera a composição mediana das bandas B2, B3, B4 e B8 sem nuvens.
composite = build_cloud_free_composite(ee, aoi_ee, strategy="cloud_score_plus")
print(f"Bandas da composição: {composite.bandNames().getInfo()}")

## Inspeção visual da composição

Renderiza a composição em um mapa interativo do `geemap`, quando o extra geoespacial estiver disponível, para conferência visual da cobertura e da ausência de nuvens.

In [ ]:
# A visualização é opcional: requer o extra geoespacial instalado.
try:
    import geemap

    m = geemap.Map()
    m.centerObject(aoi_ee, 10)
    m.addLayer(composite, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "S2")
    display(m)
except ImportError:
    print("geemap indisponível: inspeção visual ignorada.")

## Exportação da composição para o Google Drive

Dispara a exportação da composição como GeoTIFF para o Google Drive, com descrição determinística derivada da configuração e recorte pela área de estudo.

In [ ]:
from src.data.gee_client import export_image_to_drive, make_export_description

# Descrição determinística e tarefa de exportação da composição.
description = make_export_description(
    CONFIG.get("gee.export_prefix"),
    CONFIG.get("aoi.region_code"),
    CONFIG.get("gee.start_date"),
    CONFIG.get("gee.end_date"),
)
task = export_image_to_drive(
    ee,
    composite,
    description=description,
    region=aoi_ee,
    file_name_prefix=description,
)
print(f"Tarefa iniciada: {description} | id={task.id}")

## Acompanhamento das tarefas de exportação

Lista o estado das tarefas do Earth Engine para confirmar a conclusão das exportações antes de avançar para a próxima fase.

In [ ]:
import time

# Consulta o estado das tarefas recentes até todas concluírem.
while True:
    statuses = [t.status() for t in ee.batch.Task.list()[:5]]
    states = [s.get("state") for s in statuses]
    print(f"Estados: {states}")
    if all(state in {"COMPLETED", "FAILED", "CANCELLED"} for state in states):
        break
    time.sleep(30)